# RQ2 construct-building (part A)

Regenerates the **construct diffs / per-patch frames** and the **behavior / validation
scope** that `02_refactoring_effectiveness` and the RQ2/RQ3 table generators consume. **Run this before
`02_refactoring_effectiveness`.** Writes under `data/artifacts/constructs/` (per-patch frames, construct trees,
YAML diffs) and `artifacts/validation/` (behavior index, manual-validation scope,
summaries).

- Construct derivation is **schema-free** (paper catalogue), via `utils.construct_diff`;
  schema validity (SVR signal `n_invalid_added`) is the only schema touch,
  `utils.schema_validity` against `data/github-workflow.json`.
- Scope is behavior-based: 570 n15 GT pairs × 4 models × 2 designs × rep1 = **4,560**
  outputs; GT-positives are all in scope, GT-negatives only when the YAML changed.

In [1]:
from utils import constructs, validation_scope
from utils.dataloader import ROOT, DATA_ROOT

GT = ROOT / "data" / "ground_truth_n15.json"
OUT = ROOT / "data" / "artifacts" / "constructs"
ROOTS = [p for p in sorted(DATA_ROOT.glob("output-*")) if "gemma" not in p.name]  # gemma4 excluded
print("corpus roots:", [p.name for p in ROOTS])

corpus roots: ['output-final-deepseek', 'output-final-gemini-n15', 'output-final-gpt-n15', 'output-final-haiku-n15']


## Construct diffs + per-patch frames

In [2]:
gt = constructs.load_ground_truth(GT)
originals = constructs.load_originals(GT)
frames, written = constructs.build(results_roots=ROOTS, originals=originals, csv_dir=OUT, gt=gt)
print(f"per-patch index : {len(frames.per_patch_index):,}")
print(f"construct occ.  : {len(frames.occ):,}")
print(f"trees / diffs   : {frames.n_trees:,} / {frames.n_yaml_diffs:,}")
print(f"skipped (unparseable): {frames.skipped}")
for name, path in written.items():
    print("  wrote", path.relative_to(ROOT))

per-patch index : 3,995
construct occ.  : 8,302
trees / diffs   : 3,974 / 3,986
skipped (unparseable): 1138
  wrote data/artifacts/constructs/construct_per_patch.csv
  wrote data/artifacts/constructs/construct_per_patch_index.csv
  wrote data/artifacts/constructs/unparseable_patches.csv
  wrote data/artifacts/constructs/yaml_diffs
  wrote data/artifacts/constructs/construct_trees


## Validation scope + behavior index

In [3]:
behavior, manual, summary = validation_scope.build_scope_artifacts()
print(summary.to_string(index=False))
print(f"\nbehavior index: {len(behavior):,} outputs   |   manual scope: {len(manual):,}")

                                metric  value
                rep1_gt_scoped_outputs   4560
                  manual_scope_outputs   2432
         manual_scope_positive_outputs   2280
manual_scope_modified_negative_outputs    152
            manual_scope_existing_tree   2409
            manual_scope_existing_diff   2432
         manual_scope_unparseable_tree     23

behavior index: 4,560 outputs   |   manual scope: 2,432


## Behavior-outcome summary

Group count over `(population, behavior_outcome_by_gt)` of the full behavior index —
the small table `02_refactoring_effectiveness` reads as `behavior_summary_rep1.csv`.

In [4]:
behavior_summary = (
    behavior.groupby(["population", "behavior_outcome_by_gt"]).size()
    .reset_index(name="n")
    .sort_values(["population", "behavior_outcome_by_gt"]).reset_index(drop=True)
)
behavior_summary.to_csv(ROOT / "data" / "artifacts" / "validation" / "behavior_summary_rep1.csv", index=False)
behavior_summary

,population,behavior_outcome_by_gt,n
0,negative,correctly_not_modified,2128
1,negative,unexpected_modification,152
2,positive,expected_modified,1873
3,positive,missed_required_modification,407
